# 11.2 Spark UI Deep Dive

## Learning Objectives

By the end of this lesson you will be able to:

- Understand why Spark UI is important
- Navigate the Spark UI interface
- Analyze Jobs Tab
- Analyze Stages Tab
- Analyze Executors Tab
- Analyze SQL Tab
- Identify common performance bottlenecks
- Begin debugging slow Spark jobs

> **Core Rule:** Spark UI is one of the most important tools for a Data Engineer working with Apache Spark.

## Setup: Accessing the Spark UI

Before we explain the UI, let's start a local Spark session. 

When you run the cell below, PySpark will give you a **clickable link** to your local Spark UI. Open it in a new tab so you can monitor it as we run code!

In [6]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Initialize Spark session
spark = SparkSession.builder \
    .appName("Spark_UI_Deep_Dive") \
    .master("local[*]") \
    .getOrCreate()

# Fetch the dynamically assigned Spark UI URL
ui_url = spark.sparkContext.uiWebUrl
print("="*50)
print(f"🔥 SPARK UI IS RUNNING AT: {ui_url}")
print("CLICK THE LINK ABOVE TO OPEN THE DASHBOARD!")
print("="*50)

🔥 SPARK UI IS RUNNING AT: http://10.80.212.34:4040
CLICK THE LINK ABOVE TO OPEN THE DASHBOARD!


# Why Do We Need Spark UI?

Imagine your Spark job takes:

- 30 seconds yesterday
- 20 minutes today

The question becomes: **Why?**

Without visibility into execution, debugging is difficult. Spark UI provides detailed information about:

- Job execution
- Stage execution
- Task execution
- Resource utilization
- Query plans

It helps engineers find bottlenecks and optimize jobs.

# What is Spark UI?

Spark UI is a web-based monitoring dashboard. Whenever a Spark application runs, Spark generates an interface showing:

- Jobs
- Stages
- Executors
- Storage
- SQL Queries
- Environment Information

Think of Spark UI as the **"control room"** of your Spark application.

<h3>Spark UI Overview</h3>

<img src="../assets/Module_10/11_02_01.png" alt="Spark UI Overview" width="75%">

<p><i><b>Image Prompt:</b> Apache Spark UI dashboard overview showing top navigation tabs including Jobs, Stages, Executors, Storage, SQL and Environment. Clean educational screenshot-style illustration. White background.</i></p>

# 1. Jobs Tab Introduction

The Jobs Tab provides a high-level overview of application execution.

A **Job** represents a complete Spark action. Examples of actions include:
- `count()`
- `collect()`
- `show()`
- `write()`

Every action triggers a new Spark Job. Let's trigger some right now.

In [7]:
# Let's create some jobs for the UI
print("Generating Jobs...")
df = spark.range(1, 1000000)

# Action 1 triggers Job 0
print("Executing count()...")
df.count()

# Action 2 triggers Job 1
print("Executing take()...")
df.take(5)

print("\nDone! Go to your Spark UI and click the 'Jobs' tab. You should see multiple Completed Jobs.")

Generating Jobs...
Executing count()...


Executing take()...

Done! Go to your Spark UI and click the 'Jobs' tab. You should see multiple Completed Jobs.


# Information Available in Jobs Tab

For every job, Spark UI shows:

- Job ID
- Description
- Submission Time
- Duration
- Number of Stages
- Completion Status

This allows engineers to quickly identify the slowest job.

<h3>Jobs Tab Example</h3>
<img src="../assets/Module_10/11_02_02.png" alt="Jobs Tab" width="75%">
<p><i><b>Image Prompt:</b> Apache Spark Jobs Tab showing multiple jobs with Job ID, Duration, Status, and Stage progress bars. One job highlighted as significantly slower than others. Educational monitoring dashboard.</i></p>

# 2. Stages Tab Introduction

Every Spark Job is divided into multiple **stages**. 

A Stage contains a collection of tasks that can execute in parallel. Stages are created whenever Spark encounters a **shuffle boundary** (moving data across the network).

Let's write a query that forces a shuffle, which will create multiple stages.

In [8]:
import time

print("Running a job with a Shuffle boundary...")

# Add a random key so we can group by it
df_shuffle = df.withColumn("random_key", (F.rand() * 10).cast("int"))

# groupBy forces a shuffle, breaking this Job into multiple Stages
df_shuffle.groupBy("random_key").count().collect()

print("\nDone! Go to your Spark UI, click the 'Jobs' tab, then click on the newest Job ID.")
print("You will see it is split into 2 or more Stages. Click on the 'Stages' tab to see Shuffle Read/Write.")

Running a job with a Shuffle boundary...

Done! Go to your Spark UI, click the 'Jobs' tab, then click on the newest Job ID.
You will see it is split into 2 or more Stages. Click on the 'Stages' tab to see Shuffle Read/Write.


# Detecting Shuffle Problems

A common performance issue is excessive shuffle.

**Indicators in the Stages Tab:**
- High Shuffle Read (GBs or TBs of data)
- High Shuffle Write
- Long Stage Duration

Large shuffle operations often occur during `groupBy()`, `join()`, `distinct()`, and `orderBy()`.

<h3>Stages Tab Overview</h3>
<img src="../assets/Module_10/11_02_03.png" alt="Stages Tab" width="75%">
<p><i><b>Image Prompt:</b> Apache Spark Stages Tab displaying Stage ID, task counts, duration, shuffle read and shuffle write metrics. One stage highlighted with very high shuffle activity. Educational infographic.</i></p>

# Understanding Task Distribution & Skew

Within each stage, Spark creates **tasks**. Ideally, all tasks should finish around the same time.

If you see in the UI that 199 tasks finish in 1 second, but 1 task takes 10 minutes:
**You have Data Skew.**

<h3>Data Skew Example</h3>
<img src="../assets/Module_10/11_02_04.png" alt="Data Skew" width="75%">
<p><i><b>Image Prompt:</b> Spark Stage visualization showing many tasks finishing quickly while one task runs significantly longer. Illustration of data skew causing performance issues. White background.</i></p>

# 3. Executors Tab Introduction

Executors are worker processes responsible for executing tasks. The **Executors Tab** provides visibility into resource utilization.

It helps answer:
- Which executor is busy?
- Is memory usage too high?
- Are tasks evenly distributed across machines?

Let's put some data into memory and look at the Executors tab.

In [9]:
print("Caching data into Executor Memory...")
df_cache = spark.range(1, 10000000).withColumn("payload", F.rand())

# The action triggers the cache execution
df_cache.cache().count()

print("\nDone! Click the 'Executors' tab in the Spark UI.")
print("Look at the 'Storage Memory' column. You will see memory is now being consumed!")

Caching data into Executor Memory...



Done! Click the 'Executors' tab in the Spark UI.
Look at the 'Storage Memory' column. You will see memory is now being consumed!


# Identifying Memory Problems

Suppose one executor shows:
- Very high memory usage compared to others
- Frequent disk spills (writing to disk because RAM is full)
- Slow task completion

This indicates a memory bottleneck.

<h3>Executors Tab Metrics</h3>
<img src="../assets/Module_10/11_02_05.png" alt="Executors Tab" width="75%">
<p><i><b>Image Prompt:</b> Apache Spark Executors Tab showing executor IDs, active tasks, memory consumption, shuffle metrics and task counts. One executor highlighted with unusually high memory usage. Educational dashboard illustration.</i></p>

# 4. SQL Tab Introduction

When using DataFrames and Spark SQL, Spark generates SQL execution plans internally.

The **SQL Tab** provides visibility into:
- Query execution durations
- Physical Query Plans (Visual Graph)
- Operator performance (Joins, Filters, Scans)

Let's run a complex query to generate a beautiful execution graph in the UI.

In [10]:
print("Executing a complex query (Join + Filter + Aggregate)...")

df1 = spark.range(1, 100000).withColumnRenamed("id", "user_id")
df2 = spark.range(1, 100000).withColumn("user_id", F.col("id") * 2)

# Build the transformation
result = df1.join(df2, "user_id", "inner") \
            .filter(df2.id > 100) \
            .groupBy("user_id") \
            .count()

# Trigger execution
result.collect()

print("\nDone! Click the 'SQL' tab in Spark UI.")
print("Click on the top description link. You will see a beautiful blue graph showing the Exact execution plan (Joins, Filters, HashAggregates)!")

Executing a complex query (Join + Filter + Aggregate)...

Done! Click the 'SQL' tab in Spark UI.
Click on the top description link. You will see a beautiful blue graph showing the Exact execution plan (Joins, Filters, HashAggregates)!


# Reading Query Plans

A query plan shows the operations Spark performs. Examples include:
- `Scan` (Reading data)
- `Filter` (Applying WHERE clauses)
- `SortMergeJoin` or `BroadcastHashJoin` (Joining data)
- `HashAggregate` (GroupBy operations)

Understanding query plans helps identify expensive operators, which becomes very important when optimizing joins.

<h3>SQL Tab Overview</h3>
<img src="../assets/Module_10/11_02_06.png" alt="SQL Tab" width="75%">
<p><i><b>Image Prompt:</b> Apache Spark SQL Tab displaying SQL query execution details, physical execution plan tree, operator metrics and execution duration. Educational Spark monitoring interface.</i></p>

# Typical Debugging Workflow

When a Spark job is slow, follow this exact workflow:

1. **Open Jobs Tab:** Identify the slow job.
2. **Open Stages Tab:** Look inside that job to find the expensive stage.
3. **Check Executors Tab:** Check if one machine is running out of memory.
4. **Analyze SQL Tab:** Review the execution plan to see if a Join or GroupBy is causing the issue.

<h3>Spark UI Investigation Flow</h3>
<img src="../assets/Module_10/11_02_08.png" alt="Workflow" width="75%">
<p><i><b>Image Prompt:</b> Spark performance troubleshooting workflow diagram. Jobs Tab → Stages Tab → Executors Tab → SQL Tab leading to Root Cause Analysis and Optimization. Modern educational infographic. White background.</i></p>

# Key Takeaways

- Spark UI is the primary monitoring tool for Spark applications.
- **Jobs Tab** identifies slow jobs.
- **Stages Tab** identifies expensive stages and shuffle operations.
- **Executors Tab** reveals resource bottlenecks.
- **SQL Tab** helps analyze query execution plans.
- Effective optimization begins with proper investigation.

---

# Next Lesson: 11.3 Understanding Shuffle

In the Stages Tab, we saw that **Shuffle** is often the cause of slow jobs. 
In the next lesson we will explore:
- What exactly is Shuffle?
- Why is Shuffle so expensive?
- Shuffle Read vs Shuffle Write
- Techniques to reduce shuffle overhead